In [1]:
# 1. Import Libraries
import os
import pickle
import numpy as np
from tqdm import tqdm
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical, plot_model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add

C:\Users\lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
# 2. Define Directories
BASE_DIR = './dataset'
WORKING_DIR = './models'

In [3]:
# 3. Extract Image Features using VGG16
# We use VGG16 to convert images into math vectors
model = VGG16()
model = Model(inputs=model.inputs, outputs=model.layers[-2].output) # Remove classification layer
print(model.summary())

features = {}
directory = os.path.join(BASE_DIR, 'Images')
# This loop might take 10-20 mins depending on your GPU/CPU
for img_name in tqdm(os.listdir(directory)):
    img_path = directory + '/' + img_name
    image = load_img(img_path, target_size=(224, 224))
    image = img_to_array(image)
    image = image.reshape((1, image.shape[0], image.shape[1], image.shape[2]))
    image = preprocess_input(image)
    feature = model.predict(image, verbose=0)
    image_id = img_name.split('.')[0]
    features[image_id] = feature

# Save features (Optional backup)
pickle.dump(features, open(os.path.join(WORKING_DIR, 'features.pkl'), 'wb'))

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block1_conv1 (Conv2D)                │ (None, 224, 224, 64)        │           1,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block1_conv2 (Conv2D)                │ (None, 224, 224, 64)        │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block1_pool (MaxPooling2D)           │ (None, 112, 112, 64)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block2_conv1 (Conv2D)                │ (None, 112, 112, 128)       │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block2_conv2 (Conv2D)                │ (None, 112, 112, 128)       │         147,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block2_pool (MaxPooling2D)           │ (None, 56, 56, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_conv1 (Conv2D)                │ (None, 56, 56, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_conv2 (Conv2D)                │ (None, 56, 56, 256)         │         590,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_conv3 (Conv2D)                │ (None, 56, 56, 256)         │         590,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_pool (MaxPooling2D)           │ (None, 28, 28, 256)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_conv1 (Conv2D)                │ (None, 28, 28, 512)         │       1,180,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_conv2 (Conv2D)                │ (None, 28, 28, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_conv3 (Conv2D)                │ (None, 28, 28, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_pool (MaxPooling2D)           │ (None, 14, 14, 512)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_conv1 (Conv2D)                │ (None, 14, 14, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_conv2 (Conv2D)                │ (None, 14, 14, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_conv3 (Conv2D)                │ (None, 14, 14, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_pool (MaxPooling2D)           │ (None, 7, 7, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 25088)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ fc1 (Dense)                          │ (None, 4096)                │     102,764,544 │
├──────────────────────────────────────┼─────────────────────────────┼──────────────

 Total params: 134,260,544 (512.16 MB)

 Trainable params: 134,260,544 (512.16 MB)

 Non-trainable params: 0 (0.00 B)

None


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8091/8091 [25:26<00:00,  5.30it/s]


In [4]:
# 4. Load Captions
with open(os.path.join(BASE_DIR, 'captions.txt'), 'r') as f:
    next(f) # Skip header
    captions_doc = f.read()

# Map image_id to list of captions
mapping = {}
for line in tqdm(captions_doc.split('\n')):
    tokens = line.split(',')
    if len(tokens) < 2: continue
    image_id, caption = tokens[0], tokens[1:]
    image_id = image_id.split('.')[0]
    caption = " ".join(caption)
    if image_id not in mapping:
        mapping[image_id] = []
    mapping[image_id].append(caption)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 40456/40456 [00:00<00:00, 1041547.56it/s]


In [5]:
# 5. Preprocess Captions
def clean(mapping):
    for key, captions in mapping.items():
        for i in range(len(captions)):
            caption = captions[i]
            caption = caption.lower()
            caption = caption.replace('[^A-Za-z]', '') # Remove special chars
            caption = caption.replace('\s+', ' ')      # Remove extra spaces
            # Add start and end tags for LSTM
            caption = 'startseq ' + " ".join([word for word in caption.split() if len(word)>1]) + ' endseq'
            captions[i] = caption

clean(mapping)

# Collect all words
all_captions = []
for key in mapping:
    for caption in mapping[key]:
        all_captions.append(caption)

# Tokenize Text
tokenizer = Tokenizer()
tokenizer.fit_on_texts(all_captions)
vocab_size = len(tokenizer.word_index) + 1
max_length = max(len(c.split()) for c in all_captions)

# Save the tokenizer (CRITICAL for the app)
pickle.dump(tokenizer, open(os.path.join(WORKING_DIR, 'tokenizer.pkl'), 'wb'))

In [6]:
# 6. Data Generator (To handle large RAM usage)
def data_generator(data_keys, mapping, features, tokenizer, max_length, vocab_size, batch_size):

    while True:
        X1, X2, y = [], [], []
        
        for key in data_keys:
            captions = mapping[key]
            feature = features[key][0]

            for caption in captions:
                seq = tokenizer.texts_to_sequences([caption])[0]

                for i in range(1, len(seq)):
                    in_seq, out_seq = seq[:i], seq[i]

                    in_seq = pad_sequences([in_seq], maxlen=max_length)[0]
                    out_seq = to_categorical(out_seq, num_classes=vocab_size)

                    X1.append(feature)
                    X2.append(in_seq)
                    y.append(out_seq)

                    # Proper batch check
                    if len(X1) == batch_size:
                        yield (
                            np.array(X1, dtype=np.float32),
                            np.array(X2, dtype=np.int32)
                        ), np.array(y, dtype=np.float32)

                        X1, X2, y = [], [], []

In [7]:
# 7. Define Model Architecture
# Image Feature Extractor
inputs1 = Input(shape=(4096,))
fe1 = Dropout(0.4)(inputs1)
fe2 = Dense(256, activation='relu')(fe1)

# Sequence Processor (LSTM)
inputs2 = Input(shape=(max_length,))
se1 = Embedding(vocab_size, 256, mask_zero=True)(inputs2)
se2 = Dropout(0.4)(se1)
se3 = LSTM(256)(se2)

# Decoder
decoder1 = add([fe2, se3])
decoder2 = Dense(256, activation='relu')(decoder1)
outputs = Dense(vocab_size, activation='softmax')(decoder2)

model = Model(inputs=[inputs1, inputs2], outputs=outputs)
model.compile(loss='categorical_crossentropy', optimizer='adam')

In [8]:
# 8. Train the Model
import tensorflow as tf

batch_size = 32
steps = len(mapping) // batch_size  # number of batches per epoch

# Define the output signature to match:
# yield (X1, X2), y
output_signature = (
    (
        tf.TensorSpec(shape=(None, 4096), dtype=tf.float32),      # Image features
        tf.TensorSpec(shape=(None, max_length), dtype=tf.int32)   # Text sequences
    ),
    tf.TensorSpec(shape=(None, vocab_size), dtype=tf.float32)     # Target word
)

# Create dataset
train_dataset = tf.data.Dataset.from_generator(
    lambda: data_generator(
        list(mapping.keys()),
        mapping,
        features,
        tokenizer,
        max_length,
        vocab_size,
        batch_size
    ),
    output_signature=output_signature
)

# (Optional but recommended)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

# Train model
model.fit(
    train_dataset,
    epochs=20,
    steps_per_epoch=steps,
    verbose=1
)

Epoch 1/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 19s 68ms/step - loss: 6.1948
Epoch 2/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 17s 69ms/step - loss: 5.7204
Epoch 3/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 17s 68ms/step - loss: 5.5142
Epoch 4/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 17s 69ms/step - loss: 5.3722
Epoch 5/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 17s 68ms/step - loss: 5.1505
Epoch 6/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 17s 67ms/step - loss: 5.1055
Epoch 7/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 17s 67ms/step - loss: 4.9278
Epoch 8/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 17s 66ms/step - loss: 4.8713
Epoch 9/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 17s 67ms/step - loss: 4.7360
Epoch 10/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 17s 67ms/step - loss: 4.4858
Epoch 11/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 17s 68ms/step - loss: 4.6269
Epoch 12/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 17s 68ms/step - loss: 4.6577
Epoch 13/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 17s 66ms/step - loss: 4.5850
Epoch 14/20
252/252 ━━━━━━━━━━━━━━━━━━━━ 17s 67ms/step - loss: 4.5665
Epoch 15/20
252/252 ━━━━━━━━━

In [9]:
# 9. Save the Final Model (CRITICAL for the app)
model.save(os.path.join(WORKING_DIR, 'model.h5'))
print("Model saved successfully!")

Model saved successfully!
